# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 4, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [4]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type


## 4. Schema evolution (opt-in)

In [5]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [6]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The final schema-enforcement flag is hardcoded; inspect the actual error
from the bad-write cell rather than treating that PASS line as proof.

In [7]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": True,   # placeholder; inspect the bad-write output
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits
  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.


## Bằng chứng `_delta_log/` (do học viên thêm)
Liệt kê các commit JSON và in nội dung commit đầu tiên (v0) và commit schema-merge cuối cùng.

In [8]:
import json
from pathlib import Path
log_dir = Path(table_path) / "_delta_log"
commits = sorted(log_dir.glob("*.json"))
print(f"{log_dir}")
for f in commits:
    print(f"  {f.name}  {f.stat().st_size:>5} B")
for f in (commits[0], commits[-1]):
    print()
    print(f"===== {f.name} =====")
    for line in f.read_text(encoding="utf-8").splitlines():
        action = json.loads(line)
        print(json.dumps(action, ensure_ascii=False)[:400])

C:\Users\VUHUY\Desktop\labvinuni20k\K4-Track02-Day18-NguyenVuHuy-2A202602662-Lakehouse-Lab\_lakehouse\scratch\users_delta\_delta_log
  00000000000000000000.json   1409 B
  00000000000000000001.json   1465 B

===== 00000000000000000000.json =====
{"commitInfo": {"timestamp": 1791104477375, "operation": "WRITE", "operationParameters": {"mode": "Overwrite"}, "engineInfo": "delta-rs:py-1.6.6", "operationMetrics": {"num_added_files": 1, "num_removed_files": 0, "num_partitions": 0, "num_added_rows": 3, "execution_time_ms": 4, "num_retries": 0}, "clientVersion": "delta-rs.py-1.6.6"}}
{"protocol": {"minReaderVersion": 1, "minWriterVersion": 2}}
{"metaData": {"id": "7e5e597b-a77d-4ac9-b352-14c4c9d454e5", "name": null, "description": null, "format": {"provider": "parquet", "options": {}}, "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"name\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"age\

## Giải thích kết quả (học viên)
| Tiêu chí | Kết quả đo | Ngưỡng |
|---|---|---|
| `_delta_log/` có commit JSON | `000…000.json` (v0 `WRITE`: `commitInfo` + `protocol` + `metaData` + `add` kèm stats min/max) và `000…001.json` (append có schema merge) | ≥ 2 commit |
| Schema enforcement | `Cast error: Cannot cast string 'thirty' to value of Int64 type` — lần ghi bị chặn | bị chặn |
| Schema evolution | `schema_mode="merge"` thêm cột `tier`; DuckDB thấy 2 nhóm `[('premium', 1), (None, 3)]` | có `tier` |

**Cơ chế.** Bảng Delta = file Parquet + transaction log. Mỗi commit là một file JSON
`_delta_log/000…N.json` liệt kê action `add`/`remove`/`metaData`; reader dựng lại trạng thái bảng
bằng cách replay log, nên "bảng" là log chứ không phải thư mục. Schema của bảng nằm trong action
`metaData`, writer so schema của batch mới với schema đó **trước khi commit**: `age` là `Int64` nên
chuỗi `'thirty'` không cast được và cả transaction bị hủy — không có file nào được thêm vào log,
reader không bao giờ thấy dữ liệu bẩn. Bằng chứng: log chỉ có **2** file JSON (v0 và v1 merge-schema);
lần ghi sai kiểu nằm giữa hai lần đó nhưng không để lại commit nào. Evolution thì ngược lại: phải **opt-in** bằng
`schema_mode="merge"`, khi đó commit ghi một `metaData` mới có thêm cột `tier`; 3 dòng cũ không bị
viết lại, chỉ đọc ra `NULL` cho `tier` — vì vậy DuckDB thấy 2 nhóm (`premium` và `None`).

*Lưu ý:* dòng `[PASS] schema enforcement blocked bad write` ở cell cuối là cờ cố định trong notebook;
bằng chứng thực là thông báo lỗi `Cast error …` ở cell ghi sai kiểu phía trên.
